# Warren Truss Bridge Analysis

This notebook performs a structural analysis of a Warren Truss bridge, comparing different numerical methods for solving the linear system $Ax = b$ that describes the static equilibrium of the structure.

## 1. Setup and Bridge Generation
We use the `truss.py` module to generate the geometry and the system of equations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from matplotlib.lines import Line2D
from truss import generate_bridge_system
from solvers import gauss_no_pivoting, gauss_partial_pivoting, solve_sparse

plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['figure.dpi'] = 100

## 2. Geometric Visualization
Let's visualize the bridge skeleton before applying loads.

In [ ]:
NUM_SECTIONS = 25
A, b, nodes, members = generate_bridge_system(NUM_SECTIONS)

plt.figure(figsize=(12, 4))
for (i, j) in members:
    p1, p2 = nodes[i], nodes[j]
    plt.plot([p1[0], p2[0]], [p1[1], p2[1]], 'k-', lw=1, alpha=0.5)

plt.plot([n[0] for n in nodes], [n[1] for n in nodes], 'ko', markersize=3)
plt.title(f"Warren Truss Geometry - {NUM_SECTIONS} Sections")
plt.xlabel("Length (m)")
plt.ylabel("Height (m)")
plt.axis('equal')
plt.grid(True)
plt.show()

## 3. Numerical Benchmarking
We compare Naive Gauss, Gauss with Partial Pivoting, and NumPy's implementation.

In [ ]:
NUM_SECTIONS_BENCH = 50
A_bench, b_bench, _, _ = generate_bridge_system(NUM_SECTIONS_BENCH)

print(f"System size: {A_bench.shape[0]}x{A_bench.shape[1]}\n")
print(f"{'Method':<25} | {'Time (s)':<12} | {'Residual':<15} | {'Error vs NumPy'}")
print("-" * 80)

# 1. NumPy
t0 = time.perf_counter()
x_ref = np.linalg.solve(A_bench, b_bench)
t_np = time.perf_counter() - t0
res_np = np.linalg.norm(A_bench @ x_ref - b_bench)
print(f"{'NumPy (Standard)':<25} | {t_np:.5f}      | {res_np:.2e}      | {'-'}")

# 2. Gauss with Partial Pivoting
t0 = time.perf_counter()
x_gpp = gauss_partial_pivoting(A_bench, b_bench)
t_gpp = time.perf_counter() - t0
res_gpp = np.linalg.norm(A_bench @ x_gpp - b_bench)
err_gpp = np.linalg.norm(x_gpp - x_ref)
print(f"{'Gauss Partial Pivoting':<25} | {t_gpp:.5f}      | {res_gpp:.2e}      | {err_gpp:.2e}")

# 3. Naive Gauss
try:
    t0 = time.perf_counter()
    x_naive = gauss_no_pivoting(A_bench, b_bench)
    t_naive = time.perf_counter() - t0
    res_naive = np.linalg.norm(A_bench @ x_naive - b_bench)
    err_naive = np.linalg.norm(x_naive - x_ref)
    print(f"{'Gauss Naive':<25} | {t_naive:.5f}      | {res_naive:.2e}      | {err_naive:.2e}")
except ValueError as e:
    print(f"{'Gauss Naive':<25} | FAILED: {e}")

## 4. Sparsity Analysis
The matrix $A$ is very sparse. Let's analyze its density and memory efficiency.

In [ ]:
non_zeros = np.count_nonzero(A)
total = A.size
sparsity = (1 - non_zeros / total) * 100

print(f"Matrix Sparsity: {sparsity:.2f}%")

mem_dense = A.nbytes
mem_sparse = non_zeros * (8 + 4 + 4) # float64 + int32 + int32
print(f"Dense memory: {mem_dense} bytes")
print(f"Sparse memory (est.): {mem_sparse} bytes")

# Benchmark Sparse Solver
t0 = time.perf_counter()
x_sparse = solve_sparse(A_bench, b_bench)
t_sparse = time.perf_counter() - t0
print(f"Sparse solver time: {t_sparse:.5f} s")

## 5. Force Visualization
Red colors represent **Compression** (negative force), and Blue colors represent **Tension** (positive force). The thickness of the lines is proportional to the magnitude of the force.

In [ ]:
x_sol = gauss_partial_pivoting(A, b)
forces = x_sol[:len(members)]
max_f = np.max(np.abs(forces))

plt.figure(figsize=(16, 6))
for k, (i, j) in enumerate(members):
    xi, yi = nodes[i]
    xj, yj = nodes[j]
    f = forces[k]
    
    # Sign convention: Tension > 0 (Blue), Compression < 0 (Red)
    color = 'red' if f < 0 else 'blue'
    lw = 1 + 5 * (abs(f) / max_f)
    plt.plot([xi, xj], [yi, yj], color=color, linewidth=lw)

legend_elements = [
    Line2D([0], [0], color='blue', lw=2, label='Tension (Positive)'),
    Line2D([0], [0], color='red', lw=2, label='Compression (Negative)')
]
plt.legend(handles=legend_elements)
plt.title("Warren Truss Force Analysis")
plt.axis('equal')
plt.grid(alpha=0.3)
plt.show()
